In [ ]:
from pathlib import Path
from torch.utils.data import DataLoader
from rf_learning_dataset import RFLearningDataset

EXP = {
    "experiment_name": "tiny_baseline_cond_nonpoint_full_32",
    "project_root": "/home/liujia/RF_Image",

    "include_categories": ["carotid", "muscle", "phantom"],

    "batch_size": 16,
    "num_epochs": 100,
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "normalize": True,
    "abs_weight": 0.1,

    "model_name": "tiny_baseline_conditioned",
    "hidden": 64,
    "seed": 20260522,
}

PROJECT_ROOT = Path(EXP["project_root"])
DATA_ROOT = PROJECT_ROOT / "Data"

CKPT_DIR = PROJECT_ROOT / "checkpoint" / EXP["experiment_name"]
METRIC_DIR = PROJECT_ROOT / "test_metrics" / EXP["experiment_name"]
VIS_DIR = PROJECT_ROOT / "vis_best_model" / EXP["experiment_name"]

CKPT_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)
VIS_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT :", DATA_ROOT)
print("CKPT_DIR  :", CKPT_DIR)
print("METRIC_DIR:", METRIC_DIR)
print("VIS_DIR   :", VIS_DIR)

train_set = RFLearningDataset(
    root_dir=DATA_ROOT / "train",
    sample_group="/sample_000001",
    normalize=EXP["normalize"],
    include_categories=EXP["include_categories"],
)

val_set = RFLearningDataset(
    root_dir=DATA_ROOT / "val",
    sample_group="/sample_000001",
    normalize=EXP["normalize"],
    include_categories=EXP["include_categories"],
)

test_set = RFLearningDataset(
    root_dir=DATA_ROOT / "test",
    sample_group="/sample_000001",
    normalize=EXP["normalize"],
    include_categories=EXP["include_categories"],
)

train_loader = DataLoader(
    train_set,
    batch_size=EXP["batch_size"],
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

val_loader = DataLoader(
    val_set,
    batch_size=EXP["batch_size"],
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

test_loader = DataLoader(
    test_set,
    batch_size=EXP["batch_size"],
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

print("Train:", len(train_set))
print("Val  :", len(val_set))
print("Test :", len(test_set))

batch = next(iter(train_loader))
print("input   :", batch["input"].shape)
print("label   :", batch["label"].shape)
print("baseline:", batch["baseline"].shape)
print("category:", batch["category"])




In [ ]:
import time
import torch

def benchmark_loader_and_compute(model, loader, device, n_batches=20):
    model.eval()

    data_times = []
    compute_times = []

    it = iter(loader)

    for i in range(n_batches):
        t0 = time.time()
        batch = next(it)
        t1 = time.time()

        x = batch["input"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)
        b = batch["baseline"].to(device, non_blocking=True)

        torch.cuda.synchronize()
        t2 = time.time()

        with torch.no_grad():
            pred = model(x, b)
            loss = torch.mean(torch.abs(pred - y))

        torch.cuda.synchronize()
        t3 = time.time()

        data_times.append(t1 - t0)
        compute_times.append(t3 - t2)

    print(f"Avg data loading time: {sum(data_times)/len(data_times):.4f} s")
    print(f"Avg GPU compute time : {sum(compute_times)/len(compute_times):.4f} s")
    print(f"Data / compute ratio : {(sum(data_times)/len(data_times)) / (sum(compute_times)/len(compute_times) + 1e-12):.2f}")

In [ ]:
benchmark_loader_and_compute(model, train_loader, device, n_batches=20)

In [ ]:
from rf_models import build_model
from rf_train_utils import (
    seed_everything,
    get_device,
    count_trainable_parameters,
    train_model_jupyter,
    plot_training_curve,
)

seed_everything(EXP["seed"])

device = get_device()
print("Device:", device)

model = build_model(
    EXP["model_name"],
    in_channels=1536,
    hidden=EXP["hidden"],
    out_channels=2,
).to(device)

print(f"Trainable parameters: {count_trainable_parameters(model) / 1e6:.3f} M")

history = train_model_jupyter(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    ckpt_dir=CKPT_DIR,
    experiment_name=EXP["experiment_name"],
    num_epochs=EXP["num_epochs"],
    lr=EXP["lr"],
    weight_decay=EXP["weight_decay"],
    abs_weight=EXP["abs_weight"],
    print_every=5,
    seed=EXP["seed"],
    config=EXP,
)

In [ ]:
import torch
import pandas as pd

from rf_models import build_model
from rf_eval_utils import (
    evaluate_full_test_set,
    summarize_test_metrics,
    save_test_summaries,
    find_worse_samples,
)

from rf_visualization import (
    visualize_model_samples,
    select_indices_from_metrics_df,
)

# ============================================================
# Load best model
# ============================================================

best_model = build_model(
    EXP["model_name"],
    in_channels=1536,
    hidden=EXP["hidden"],
    out_channels=2,
).to(device)

ckpt_path = CKPT_DIR / "best_model.pth"
ckpt = torch.load(ckpt_path, map_location=device)

best_model.load_state_dict(ckpt["model"])
best_model.eval()

print("Loaded best model")
print("  epoch       :", ckpt["epoch"])
print("  best val L1 :", ckpt["best_val_l1"])
print("  ckpt path   :", ckpt_path)

# ============================================================
# Full test evaluation
# ============================================================

df_test = evaluate_full_test_set(
    model=best_model,
    dataset=test_set,
    device=device,
    batch_size=EXP["batch_size"],
    save_csv_path=METRIC_DIR / "test_per_sample_metrics.csv",
)

overall, cat_summary = summarize_test_metrics(df_test)

save_test_summaries(
    df=df_test,
    metric_dir=METRIC_DIR,
    overall=overall,
    cat_summary=cat_summary,
    prefix="test",
)

# ============================================================
# Worse samples
# ============================================================

worse_complex = find_worse_samples(
    df_test,
    metric="complex",
    top_k=20,
)

worse_abs = find_worse_samples(
    df_test,
    metric="abs",
    top_k=20,
)

# 也保存一下，方便后面查
worse_complex.to_csv(METRIC_DIR / "test_worse_complex_top20.csv", index=False)
worse_abs.to_csv(METRIC_DIR / "test_worse_abs_top20.csv", index=False)

# ============================================================
# Visualization: random/category samples
# ============================================================

vis_random = visualize_model_samples(
    model=best_model,
    dataset=test_set,
    device=device,
    save_dir=VIS_DIR / "random_by_category",
    indices=None,
    n_per_category=3,
    categories=EXP["include_categories"],
    view="xz",
    slice_index=None,
    db_min=-60,
    show=False,
    prefix="random",
)

# ============================================================
# Visualization: worst complex samples
# ============================================================

worst_indices = select_indices_from_metrics_df(
    dataset=test_set,
    df=df_test,
    top_k=9,
    metric="complex_improvement",
    ascending=True,
)

vis_worst = visualize_model_samples(
    model=best_model,
    dataset=test_set,
    device=device,
    save_dir=VIS_DIR / "worst_complex",
    indices=worst_indices,
    view="xz",
    slice_index=None,
    db_min=-60,
    show=False,
    prefix="worst_complex",
)

print("\nEvaluation package finished.")
print("Metric dir:", METRIC_DIR)
print("Vis dir   :", VIS_DIR)
print("Random visualizations:", len(vis_random))
print("Worst visualizations :", len(vis_worst))